# NB00 — WaTEM/SEDEM on Colab: Langegracht
Reference annual erosion/sediment model and grass-strip scenario, followed by an illustrative C-factor reduction. Adapted from the official MIT-licensed `original/api.ipynb`. Inputs and outputs persist on Drive; computation is local. Run top to bottom on a CPU runtime. No CN runoff in NB00. Installation and execution require validation on Colab.

## NB00.01 — Mount Drive and define paths

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, shutil, subprocess, json, hashlib, time, platform
from datetime import datetime, timezone
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Watem')
DATA = ROOT / 'data/tutorial/langegracht'
OUTPUT = ROOT / 'outputs/tutorial'
SOFTWARE = ROOT / 'software'
for p in (DATA.parent, OUTPUT, SOFTWARE): p.mkdir(parents=True, exist_ok=True)
REV = '226a7c06ae836f9736c1a88b74b5346033a2a8d6'
REPO = Path('/content/pywatemsedem')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
WORK = Path('/content/watem_runs') / RUN_ID
WORK.mkdir(parents=True)
def run(cmd, **kwargs):
    print('+', ' '.join(map(str, cmd)), flush=True)
    return subprocess.run(list(map(str,cmd)), check=True, **kwargs)
print(ROOT, sys.version, platform.machine())

## NB00.02 — Install software and pin source
The upstream changelog specifies WaTEM/SEDEM 5.0.3 and SAGA-WaTEM 2025.6.11. Use the matching Ubuntu package, not stock SAGA. Python dependencies are resolved against the Colab interpreter rather than forcing the upstream Python 3.13 lockfile. Save actual versions in the output manifest.

In [ ]:
import urllib.request
os_release = dict(line.strip().split('=',1) for line in Path('/etc/os-release').read_text().splitlines() if '=' in line)
codename = os_release.get('VERSION_CODENAME','').strip('"')
assert platform.machine() == 'x86_64', 'Expected x86_64 Colab runtime'
assert codename in {'jammy','noble','focal','bookworm'}, f'Unsupported OS for pinned SAGA release: {codename}'
run(['apt-get','update','-qq'])
run(['apt-get','install','-y','git-lfs','saga'])
run(['git','lfs','install'])
def download(url, path):
    if not path.exists():
        tmp = path.with_suffix(path.suffix+'.part')
        urllib.request.urlretrieve(url, tmp)
        tmp.replace(path)
    return path
saga_name = f'saga-vpo_2025.6.11-1-ge95f136.{codename}_amd64.deb'
saga_deb = download('https://github.com/DOV-Vlaanderen/saga-watem/releases/download/2025.6.11/'+saga_name, SOFTWARE/saga_name)
local_deb = Path('/tmp') / saga_name
shutil.copy2(saga_deb, local_deb)
run(['apt-get','install','-y',str(local_deb)])
ws = download('https://github.com/watem-sedem/watem-sedem/releases/download/5.0.3/watem_sedem', SOFTWARE/'watem_sedem_5.0.3')
binary = Path('/usr/local/bin/watem_sedem')
shutil.copy2(ws,binary); binary.chmod(0o755)
os.environ['WATEMSEDEM'] = str(binary.parent)
# saga-vpo contains plugins, not the SAGA command-line program.
# Ubuntu's saga package supplies saga_cmd.
saga_cmd = shutil.which('saga_cmd')
if not saga_cmd:
    run(['dpkg','-L','saga'])
    raise RuntimeError('saga_cmd missing after installing saga')
assert shutil.which('watem_sedem'), 'watem_sedem missing from PATH'
print('SAGA executable:', saga_cmd)
print('WaTEM executable:', shutil.which('watem_sedem'))
plugin_files = subprocess.check_output(['dpkg','-L','saga-vpo'],text=True).splitlines()
print('Installed SAGA plugin libraries:', [p for p in plugin_files if p.endswith('.so')])
run(['saga_cmd','--version'])
for args in (['saga_cmd','topology'], ['saga_cmd','watem']):
    p = subprocess.run(args,capture_output=True,text=True)
    msg = p.stdout+p.stderr
    print(msg[:3500])
    assert 'error: select a library' not in msg.lower() and 'cannot load' not in msg.lower(), f'Missing/incompatible SAGA library: {args}'
run(['ldd',str(binary)])
if not REPO.exists():
    run(['git','clone','https://github.com/watem-sedem/pywatemsedem.git',REPO],
        env={**os.environ,'GIT_LFS_SKIP_SMUDGE':'1'})
run(['git','-C',REPO,'checkout',REV],env={**os.environ,'GIT_LFS_SKIP_SMUDGE':'1'})
run([sys.executable,'-m','pip','install',str(REPO)])
import pywatemsedem
print('pywatemsedem',pywatemsedem.__version__)

## NB00.03 — Retrieve and verify official tutorial data
Cache only the required tutorial inputs and their shapefile sidecars. Validate every Git LFS file against its SHA256 pointer; a missing or incomplete cache is rebuilt without silently using old inputs.

In [ ]:
required = ['dtm.tif','K3.tif','basemap_landuse.tif','userchoices.ini']
stems = ['catchm_langegracht','river','infrastructure','roads','water','parcels','grass_strips','buffers','force_routing','outlets']
source = REPO/'tests/data'
names = sorted(p.name for p in source.iterdir() if p.is_file() and (p.name in required or p.stem in stems))
assert set(required).issubset(names)
for stem in stems:
    assert all(stem+ext in names for ext in ('.shp','.shx','.dbf','.prj')), stem
def expected_hash(name):
    raw = subprocess.check_output(['git','-C',str(REPO),'show',f'{REV}:tests/data/{name}'])
    if raw.startswith(b'version https://git-lfs'):
        return next(x.split(':',1)[1] for x in raw.decode().splitlines() if x.startswith('oid sha256:'))
    return hashlib.sha256(raw).hexdigest()
hashes = {name:expected_hash(name) for name in names}
import urllib.request, urllib.error
from concurrent.futures import ThreadPoolExecutor, as_completed

def valid_file(path, name):
    return path.is_file() and hashlib.sha256(path.read_bytes()).hexdigest() == hashes[name]

def valid(folder):
    return all(valid_file(folder/n,n) for n in names)

DATA.mkdir(parents=True,exist_ok=True)
missing = [n for n in names if not valid_file(DATA/n,n)]
print(f'Valid cached files: {len(names)-len(missing)}/{len(names)}')
if missing:
    # Git LFS pull can fail at a detached HEAD or due to upstream LFS service.
    # Capture stderr in the notebook, rather than only showing CalledProcessError.
    pending = [n for n in missing if not valid_file(source/n,n)]
    if pending:
        p = subprocess.run(['git','-C',str(REPO),'lfs','pull','origin',
                            '--include='+','.join('tests/data/'+n for n in pending)],
                           capture_output=True,text=True)
        print('Git LFS return code:',p.returncode)
        print(p.stdout[-4000:])
        print(p.stderr[-6000:])
    def retrieve(name):
        if valid_file(DATA/name,name):
            return name,'cached'
        if valid_file(source/name,name):
            shutil.copy2(source/name,DATA/name)
            return name,'git-lfs'
        # GitHub media endpoint serves actual LFS objects rather than pointers.
        # Always require the exact SHA256 from the pinned upstream Git tree.
        errors=[]
        tmp=DATA/(name+'.part')
        for base in (
            f'https://media.githubusercontent.com/media/watem-sedem/pywatemsedem/{REV}/tests/data/',
            f'https://github.com/watem-sedem/pywatemsedem/raw/{REV}/tests/data/',
            f'https://raw.githubusercontent.com/watem-sedem/pywatemsedem/{REV}/tests/data/',
        ):
            for attempt in range(2):
                try:
                    req=urllib.request.Request(base+name,headers={'User-Agent':'Watem-Colab/1.0'})
                    with urllib.request.urlopen(req,timeout=60) as response, tmp.open('wb') as dst:
                        shutil.copyfileobj(response,dst)
                    if not valid_file(tmp,name):
                        raise ValueError('SHA256 mismatch (possibly a Git LFS pointer)')
                    tmp.replace(DATA/name)
                    return name,'direct download'
                except Exception as e:
                    errors.append(f'{base}: {type(e).__name__}: {e}')
                    tmp.unlink(missing_ok=True)
                    if attempt==0: time.sleep(2)
        raise RuntimeError(name+': '+' | '.join(errors))
    failures=[]
    with ThreadPoolExecutor(max_workers=4) as pool:
        futures={pool.submit(retrieve,name):name for name in missing}
        for f in as_completed(futures):
            try:
                print(*f.result())
            except Exception as e:
                failures.append(str(e))
                print('FAILED:',e)
    if failures:
        raise RuntimeError(f'{len(failures)} data downloads failed. Valid files are retained on Drive. See HTTP/LFS diagnostics above.')
assert valid(DATA), 'Tutorial data checksum validation failed'
(DATA/'source_manifest.json').write_text(json.dumps({'revision':REV,'sha256':hashes},indent=2))
inputdata_folder = WORK/'inputs'
inputdata_folder.mkdir(exist_ok=True)
for n in names: shutil.copy2(DATA/n,inputdata_folder/n)
print(f'Validated {len(names)} files in {DATA}')
print('Local working data:', inputdata_folder)

## NB00.04 — Inspect input maps

In [ ]:
import rasterio
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
from rasterio.plot import show
fig,axes=plt.subplots(1,3,figsize=(16,5))
for ax,name in zip(axes,['dtm.tif','K3.tif','basemap_landuse.tif']):
    with rasterio.open(inputdata_folder/name) as ds:
        print(name,ds.crs,ds.shape,ds.nodata)
        show(ds,ax=ax)
    ax.set_title(name)
plt.tight_layout(); plt.show()
parcels=gpd.read_file(inputdata_folder/'parcels.shp')
print(parcels.columns.tolist(),len(parcels))
parcels.plot(column='LANDUSE',legend=True,figsize=(7,7)); plt.show()

## NB00.05 — Initialize catchment (official tutorial parameters)

In [ ]:
from pywatemsedem.catchment import Catchment
catchment = Catchment('langegracht',inputdata_folder/'catchm_langegracht.shp',
    inputdata_folder/'dtm.tif',20,31370,-9999,results_folder=WORK)
catchment.kfactor=inputdata_folder/'K3.tif'
catchment.landuse=inputdata_folder/'basemap_landuse.tif'
catchment.vct_river=inputdata_folder/'river.shp'
catchment.vct_infrastructure_buildings=inputdata_folder/'infrastructure.shp'
catchment.vct_infrastructure_roads=inputdata_folder/'roads.shp'
catchment.vct_water=inputdata_folder/'water.shp'
catchment.dtm.plot()
catchment.segments.plot(nodata=0)

## NB00.06 — Read official model settings
No calibration or hydrological assumptions are added.

In [ ]:
from pywatemsedem.choices import Choices, Options, Parameters, Extensions, ExtensionsParameters, Output
from pywatemsedem.scenario import Scenario
default_choices=inputdata_folder/'userchoices.ini'
options=Options(); options.read_values_from_ini(default_choices)
parameters=Parameters(); parameters.read_values_from_ini(default_choices)
extensions=Extensions(); extensions.read_values_from_ini(default_choices)
output=Output(); output.read_values_from_ini(default_choices)
extensionparameters=ExtensionsParameters(extensions)
extensionparameters.apply_defaults()
for p in (extensionparameters.ktc_low,extensionparameters.ktc_high,extensionparameters.ktc_limit):
    p.read_value_from_ini(default_choices)
choices=Choices(options,parameters,extensions,extensionparameters,output)
assert not choices.extensions.curve_number.value, 'NB00 expects standard erosion model'
print(default_choices.read_text())

## NB00.07 — Prepare baseline and execute
The upstream wrapper's run_model does not check subprocess return codes. We use the same executable and generated INI, but explicitly check return status and output existence.

In [ ]:
def prepare(scenario, source_measures=False):
    scenario.composite_landuse=scenario.create_composite_landuse()
    scenario.cfactor=scenario.create_cfactor(use_source_oriented_measures=source_measures)
    ep=scenario.choices.extensionparameters
    scenario.ktc=scenario.create_ktc(ep.ktc_low.value,ep.ktc_high.value,ep.ktc_limit.value,
        not scenario.choices.extensions.create_ktc_map.value)
    scenario.prepare_input_files()
    scenario.create_ini_file()
    return scenario
def execute(scenario):
    dest=scenario.sfolder.wsoutput_folder
    assert not any(dest.iterdir()), f'Output directory already populated: {dest}; start a fresh run'
    p=subprocess.run([str(binary),str(scenario.ini)],capture_output=True,text=True,errors='replace')
    log=scenario.sfolder.scenario_folder/'model.log'
    log.write_text(p.stdout+'\n'+p.stderr)
    print(p.stdout[-5000:],p.stderr[-3000:])
    p.check_returncode()
    rasters=list(dest.glob('*.rst'))
    assert rasters, f'No output rasters: see {log}'
    print('Outputs:',sorted(x.name for x in dest.iterdir()))
    return scenario
baseline=Scenario(catchment,2019,1,choices)
baseline.vct_parcels=inputdata_folder/'parcels.shp'
prepare(baseline)
baseline.cfactor.plot()
execute(baseline)

## NB00.08 — Management scenarios
S2 reproduces the upstream grass-strip example. S4 uses the upstream illustrative 80% C-factor reduction on agricultural parcels, not an empirically established regenerative-agriculture effect. The original tutorial comments out S4 execution; here we explicitly attempt it.

In [ ]:
grass=Scenario(catchment,2019,2,choices)
grass.vct_parcels=inputdata_folder/'parcels.shp'
grass.vct_grass_strips=inputdata_folder/'grass_strips.shp'
execute(prepare(grass))
reduced=Scenario(catchment,2019,4,choices)
modified=gpd.read_file(inputdata_folder/'parcels.shp')
modified.loc[modified['LANDUSE']==-9999,'C_reduct']=0.8
reduced.vct_parcels=modified
execute(prepare(reduced,source_measures=True))
scenarios={'baseline':baseline,'grass_strips':grass,'C_reduction':reduced}

## NB00.09 — Inspect and export erosion maps
Keep native raster units and sign convention. Read the model output reports before interpreting differences as sediment delivery or avoided erosion.

In [ ]:
def find_erosion(scenario):
    files=[p for p in scenario.sfolder.wsoutput_folder.glob('*.rst') if 'watereros' in p.name.lower()]
    assert len(files)==1, f'Expected one WATEREROS raster, found {files}'
    return files[0]
maps={}
for label,s in scenarios.items():
    path=find_erosion(s)
    with rasterio.open(path) as ds:
        a=ds.read(1,masked=True)
        profile=ds.profile.copy()
        profile.update(driver='GTiff',count=1,compress='deflate')
        out=s.sfolder.scenario_folder/'watereros.tif'
        with rasterio.open(out,'w',**profile) as dst: dst.write(ds.read(1),1)
        maps[label]=a
        print(label,path.name,'min/max',float(a.min()),float(a.max()))
lim=float(np.percentile(np.abs(maps['baseline'].compressed()),98))
lim=max(lim,1e-9)
fig,axes=plt.subplots(1,3,figsize=(17,6))
for ax,(label,a) in zip(axes,maps.items()):
    im=ax.imshow(a,cmap='RdBu',vmin=-lim,vmax=lim)
    ax.set_title(label); ax.axis('off')
fig.colorbar(im,ax=axes.tolist(),label='Native WATEREROS values',shrink=.7)
plt.show()
for label,s in scenarios.items():
    print('\n',label)
    for p in sorted(s.sfolder.wsoutput_folder.glob('*.txt')):
        if 'total' in p.name.lower() or 'budget' in p.name.lower():
            print(p.name,p.read_text(errors='replace')[:4000])

## NB00.10 — Persist results and provenance
Each run gets a separate timestamped folder; no prior Drive results are deleted. Model INI files retain local execution paths and should be regenerated for reruns.

In [ ]:
manifest={'run_id':RUN_ID,'upstream_revision':REV,'data_sha256':hashes,
          'watem_sedem':'5.0.3','saga_watem':'2025.6.11','os':os_release,
          'python':sys.version,'pywatemsedem':pywatemsedem.__version__}
(WORK/'run_manifest.json').write_text(json.dumps(manifest,indent=2))
(WORK/'pip_freeze.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
dest=OUTPUT/RUN_ID
shutil.copytree(WORK,dest,ignore=shutil.ignore_patterns('inputs'))
print('Saved results:',dest)